<a href="https://colab.research.google.com/github/FlaviaVSC/ZIGURAT-M4T-1-AULA-01/blob/main/M5T2_Fl%C3%A1via_Ferreira.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
#Aluna: Flávia Ferreira
#Sistema Multi-Agente com LangGraph para Análise de Modelo IFC

#Este notebook implementa um sistema agentic baseado em LangGraph
#para análise técnica de um modelo IFC utilizando IfcOpenShell.

#Código adaptado das aulas do Prof. Antonio Cavalcanti.

In [31]:
#Envio do modelo .ifc
from google.colab import files

uploaded = files.upload()

Saving residencia_palmeiras_142_recife.ifc to residencia_palmeiras_142_recife (2).ifc


In [32]:
#Instalação das Dependências

!pip install ifcopenshell
!pip install langchain langgraph
!pip install python-docx

In [33]:
#Imports e Tipos de Estados

from typing import TypedDict, Dict, List
from langgraph.graph import StateGraph, START, END
import ifcopenshell
import ifcopenshell.util.element
from docx import Document

In [34]:
#Definição do Estado
#O estado representa a “pasta do projeto”, sendo o contrato entre todos os agentes.

class EstadoProjeto(TypedDict):
    elementos_ifc: Dict
    quantitativos: Dict
    alertas: List[str]
    relatorio_final: str

In [35]:
#Tool BIM (IfcOpenShell REAL)

def extrair_dados_ifc(caminho_ifc: str) -> Dict:
    modelo = ifcopenshell.open(caminho_ifc)

    paredes = modelo.by_type("IfcWall")
    portas = modelo.by_type("IfcDoor")
    janelas = modelo.by_type("IfcWindow")
    lajes = modelo.by_type("IfcSlab")

    area_janelas = 0.0
    for j in janelas:
        psets = ifcopenshell.util.element.get_psets(j)
        area_janelas += psets.get(
            "Qto_WindowBaseQuantities", {}
        ).get("Area", 0)

    return {
        "IfcWall": {"total": len(paredes)},
        "IfcDoor": {"total": len(portas)},
        "IfcWindow": {
            "total": len(janelas),
            "area_total_m2": round(area_janelas, 2)
        },
        "IfcSlab": {"total": len(lajes)}
    }

In [36]:
#AGENTE 1: BIM / IFC

def agente_bim_ifc(estado: EstadoProjeto) -> Dict:
    dados_ifc = extrair_dados_ifc("residencia_palmeiras_142_recife.ifc")
    return {"elementos_ifc": dados_ifc}

In [37]:
#AGENTE 2: Quantitativos

def agente_quantitativos(estado: EstadoProjeto) -> Dict:
    elementos = estado["elementos_ifc"]

    quantitativos = {
        "paredes": elementos["IfcWall"]["total"],
        "portas": elementos["IfcDoor"]["total"],
        "janelas": elementos["IfcWindow"]["total"],
        "area_janelas_m2": elementos["IfcWindow"]["area_total_m2"],
        "lajes": elementos["IfcSlab"]["total"]
    }

    return {"quantitativos": quantitativos}

In [38]:
#AGENTE 3: Alertas Técnicos Simples

def agente_alertas(estado: EstadoProjeto) -> Dict:
    alertas = []

    if estado["quantitativos"]["portas"] == 0:
        alertas.append("Nenhuma porta detectada no residencia_palmeiras_142_recife.ifc.")

    if estado["quantitativos"]["area_janelas_m2"] == 0:
        alertas.append("Área de janelas não encontrada — possível ausência de Psets.")

    return {"alertas": alertas}

In [39]:
#AGENTE 4: Relatório Profissional (.docx)

def agente_relatorio(estado: EstadoProjeto) -> Dict:
    doc = Document()
    doc.add_heading("Relatório Técnico – Análise IFC", level=1)

    doc.add_heading("Quantitativos", level=2)
    for k, v in estado["quantitativos"].items():
        doc.add_paragraph(f"{k}: {v}")

    if estado["alertas"]:
        doc.add_heading("Alertas Técnicos", level=2)
        for alerta in estado["alertas"]:
            doc.add_paragraph(f"- {alerta}")

    caminho = "output_relatorio.docx"
    doc.save(caminho)

    return {"relatorio_final": caminho}

In [40]:
#Construção do Grafo LangGraph

builder = StateGraph(EstadoProjeto)

builder.add_node("bim", agente_bim_ifc)
builder.add_node("quantitativos", agente_quantitativos)
builder.add_node("alertas", agente_alertas)
builder.add_node("relatorio", agente_relatorio)

builder.add_edge(START, "bim")
builder.add_edge("bim", "quantitativos")
builder.add_edge("quantitativos", "alertas")
builder.add_edge("alertas", "relatorio")
builder.add_edge("relatorio", END)

grafo = builder.compile()

In [41]:
#Resumo Geral

def resumo(resultado):
    q = resultado["quantitativos"]
    alertas = resultado["alertas"]

    print("RESUMO DO MODELO IFC")
    print("-" * 40)
    print(f"Paredes: {q['paredes']}")
    print(f"Portas: {q['portas']}")
    print(f"Janelas: {q['janelas']}")
    print(f"Lajes: {q['lajes']}")

    if q["area_janelas_m2"] > 0:
        print(f"Área total de janelas: {q['area_janelas_m2']} m²")
    else:
        print("Área de janelas: não informada no modelo")

    if alertas:
        print("\n⚠️ ALERTAS TÉCNICOS")
        for a in alertas:
            print(f"- {a}")

    print("\n📄 Relatório gerado em:")
    print(resultado["relatorio_final"])

In [42]:
#Execução do pipeline

resultado = grafo.invoke({})
resumo(resultado)

RESUMO DO MODELO IFC
----------------------------------------
Paredes: 24
Portas: 11
Janelas: 14
Lajes: 4
Área de janelas: não informada no modelo

⚠️ ALERTAS TÉCNICOS
- Área de janelas não encontrada — possível ausência de Psets.

📄 Relatório gerado em:
output_relatorio.docx


In [45]:
#Agente de relatório


def agente_relatorio(estado: EstadoProjeto) -> Dict:
    q = estado["quantitativos"]
    alertas = estado["alertas"]

    doc = Document()
    doc.add_heading("Relatório Simplificado do Modelo BIM", level=1)

    doc.add_paragraph(
        "Este relatório apresenta um resumo automático dos principais "
        "elementos identificados no modelo IFC analisado."
    )

    doc.add_heading("Resumo Geral", level=2)
    doc.add_paragraph(f"Paredes: {q['paredes']}")
    doc.add_paragraph(f"Portas: {q['portas']}")
    doc.add_paragraph(f"Janelas: {q['janelas']}")
    doc.add_paragraph(f"Lajes: {q['lajes']}")

    if q["area_janelas_m2"] > 0:
        doc.add_paragraph(
            f"Área total de janelas: {q['area_janelas_m2']} m²"
        )
    else:
        doc.add_paragraph(
            "Área de janelas não informada no modelo."
        )

    if alertas:
        doc.add_heading("Observações Técnicas", level=2)
        for alerta in alertas:
            doc.add_paragraph(f"- {alerta}")

    doc.add_paragraph(
        "\nRelatório gerado automaticamente por sistema multi‑agente "
        "baseado em LangGraph e IfcOpenShell."
    )

    caminho = "output_relatorio.docx"
    doc.save(caminho)

    return {"relatorio_final": caminho}

In [47]:
#Download do relatório gerado
from google.colab import files

files.download("output_relatorio.docx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>